# Lab 08: Mixture-of-Experts routing and dispatch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab08_moe_router.ipynb) · lab page: [`labs/lab08.html`](../lab08.html) · file: [`labs/lab08_moe_router.py`](../lab08_moe_router.py)

**Time:** 60-90 min

## Goal

An MoE layer replaces the one FFN of a block with E expert FFNs and a
router that sends each token to only k of them. You build the two router
types used today (softmax top-k: Mixtral, Qwen-MoE; sigmoid top-k with a
balancing bias: DeepSeek-V3), the auxiliary load-balancing loss, two ways
to dispatch tokens to experts (a simple per-expert loop, and the
sort-and-scatter layout that grouped-GEMM kernels use), and capacity-limited
dispatch with token dropping.

## The maths, step by step

```text
T tokens (batch x sequence, flattened), d width, E experts, k experts per token.

  1. Router logits  s = x W_r^T                        [T, E]
  2a. Softmax top-k (Mixtral):
        p = softmax(s)                                 [T, E]
        idx = the k largest p per token (descending)   [T, k]
        w = p[idx] / sum(p[idx])                       [T, k]   (renormalized)
  2b. Sigmoid top-k (DeepSeek-V3):
        g = sigmoid(s)                                 [T, E]   (independent scores)
        idx = top-k of (g + bias)                      bias [E] only steers the CHOICE
        w = g[idx] / sum(g[idx])                       weights use g, not g + bias
  3. Load-balancing loss (Switch / Mixtral style):
        f_e = (number of the T*k assignments that went to expert e) / (T*k)
        P_e = mean over tokens of p[:, e]
        L_aux = E * sum_e f_e * P_e
     = 1.0 when routing is perfectly uniform, up to E when everything goes to
     one expert. f is a count (no gradient); the gradient flows through P.
     (The MoE guide writes alpha * E * sum f * P; this function returns the
     unweighted E * sum f * P, so its floor is 1. Multiply by alpha ~ 0.01
     when adding it to the LM loss.)
  4. Dispatch: y[t] = sum_j w[t, j] * Expert_{idx[t, j]}(x[t])        [T, d]
       loop:    for each expert e, gather its tokens, run it once, scatter back.
       sorted:  flatten the T*k assignments, sort them by expert id, so each
                expert's tokens are one contiguous slice; run every slice,
                weight, and index_add the results back to their tokens.
  5. Capacity: each expert accepts at most
        C = ceil(capacity_factor * T * k / E)
     assignments. Served in order (token 0's choices, then token 1's, ...);
     assignments beyond C are dropped: they add nothing, and the token keeps
     only its residual for that expert.
```

## Read alongside

- [MoE guide #moe-ffn](../../moe/moe_guide.html#moe-ffn): 1. Dense FFN vs MoE FFN
- [MoE guide #router](../../moe/moe_guide.html#router): 2. The Router (2.1 load balancing, #router-viz)
- [MoE guide #inference](../../moe/moe_guide.html#inference): 5.4 capacity factor and token dropping, 5.5 grouped GEMM: from E tiny matmuls to one kernel
- [moe.py](../../moe/moe.py): TopKRouter, RoutedExperts.dispatch_loop / dispatch_sorted

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(8, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab08_moe_router.py` or the lab page. The reference solution is `labs/solutions/lab08_moe_router.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Step 1: routers

**Hints for `softmax_topk_router`** (open one at a time):

<details><summary>Hint 1</summary>

```text
torch.softmax, then torch.topk(probs, k, dim=-1) (sorted descending by default).
```

</details>

<details><summary>Hint 2</summary>

```text
weights = top_p / top_p.sum(dim=-1, keepdim=True)
```

</details>

In [ ]:
def softmax_topk_router(logits: torch.Tensor, k: int):
    """Mixtral-style router.

    logits [T, E] -> (weights [T, k], indices [T, k] int64, probs [T, E])
    probs = softmax(logits); indices = the k most probable experts, in
    DESCENDING order of probability; weights = their probs renormalized to sum to 1.
    """
    raise NotImplementedError  # TODO

**Hints for `sigmoid_topk_router`** (open one at a time):

<details><summary>Hint 1</summary>

```text
the bias only changes which experts are chosen, never the weights.
```

</details>

<details><summary>Hint 2</summary>

```text
_, indices = torch.topk(scores + bias, k); chosen = scores.gather(-1, indices)
```

</details>

In [ ]:
def sigmoid_topk_router(logits: torch.Tensor, k: int, bias: torch.Tensor = None):
    """DeepSeek-V3-style router.

    logits [T, E], bias [E] or None -> (weights [T, k], indices [T, k] int64)
    scores = sigmoid(logits); choose the k experts with the largest scores + bias
    (descending); weights = the UNBIASED scores of the chosen experts, renormalized.
    """
    raise NotImplementedError  # TODO

## Step 2: the load-balancing loss

**Hints for `load_balancing_loss`** (open one at a time):

<details><summary>Hint 1</summary>

```text
f = torch.bincount(indices.flatten(), minlength=E) / indices.numel()
```

</details>

<details><summary>Hint 2</summary>

```text
P = probs.mean(dim=0); return E * (f * P).sum()
(f must be float; the gradient only flows through P.)
```

</details>

In [ ]:
def load_balancing_loss(probs: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
    """L_aux = E * sum_e f_e * P_e  (see the module docstring).

    probs [T, E] (softmax router probabilities), indices [T, k] -> scalar tensor.
    """
    raise NotImplementedError  # TODO

## Given: experts

Given: `class Expert(nn.Module)` is already written. Run the cell.

In [ ]:
class Expert(nn.Module):
    """One expert: a small GELU MLP, d -> d_ff -> d."""

    def __init__(self, d: int, d_ff: int):
        super().__init__()
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))

Given: `make_experts(n_experts, d, d_ff)` is already written. Run the cell.

In [ ]:
def make_experts(n_experts: int, d: int, d_ff: int) -> nn.ModuleList:
    return nn.ModuleList(Expert(d, d_ff) for _ in range(n_experts))

## Step 3: dispatch, two ways

**Hints for `dispatch_loop`** (open one at a time):

<details><summary>Hint 1</summary>

```text
for e in range(len(experts)): find the (token, slot) pairs that chose e:
tok, slot = torch.where(indices == e)
```

</details>

<details><summary>Hint 2</summary>

```text
if tok.numel() == 0: continue;  out = experts[e](x[tok])   # [n_e, d]
```

</details>

<details><summary>Hint 3</summary>

```text
y.index_add_(0, tok, out * weights[tok, slot, None])
(index_add_ sums correctly even if the same token appears twice)
```

</details>

In [ ]:
def dispatch_loop(x: torch.Tensor, weights: torch.Tensor, indices: torch.Tensor,
                  experts: nn.ModuleList) -> torch.Tensor:
    """y[t] = sum_j weights[t, j] * experts[indices[t, j]](x[t]), one expert at a time.

    x [T, d], weights/indices [T, k] -> y [T, d]. Call each expert at most once,
    on all of its tokens together (skip experts that got no tokens).
    """
    raise NotImplementedError  # TODO

**Hints for `dispatch_sorted`** (open one at a time):

<details><summary>Hint 1</summary>

```text
token_of = torch.arange(T).repeat_interleave(k) matches indices.flatten().
```

</details>

<details><summary>Hint 2</summary>

```text
xs = x[token_of[order]]; chunks = torch.split(xs, counts.tolist())
```

</details>

<details><summary>Hint 3</summary>

```text
out = torch.cat([experts[e](c) for e, c in enumerate(chunks) if len(c)])  (keep
the order!) then y.index_add_(0, token_of[order], out * weight_of[order, None])
```

</details>

In [ ]:
def dispatch_sorted(x: torch.Tensor, weights: torch.Tensor, indices: torch.Tensor,
                    experts: nn.ModuleList) -> torch.Tensor:
    """Same result as dispatch_loop, via sort -> contiguous slices -> scatter.

    1. flatten the assignments: expert_of = indices.flatten() [T*k], token_of = the
       token each one belongs to, weight_of = weights.flatten()
    2. order = argsort(expert_of): now every expert's assignments are contiguous
    3. counts = bincount(expert_of, minlength=E); run expert e on its slice of
       x[token_of[order]] (this layout is what grouped-GEMM kernels consume)
    4. multiply by the weights and index_add the rows back to their tokens
    """
    raise NotImplementedError  # TODO

## Step 4: capacity and token dropping

**Hints for `expert_capacity`** (open one at a time):

<details><summary>Hint 1</summary>

```text
math.ceil (import math at the top), then int().
```

</details>

In [ ]:
def expert_capacity(n_tokens: int, n_experts: int, k: int, capacity_factor: float) -> int:
    """C = ceil(capacity_factor * n_tokens * k / n_experts), as an int."""
    raise NotImplementedError  # TODO

**Hints for `dispatch_with_capacity`** (open one at a time):

<details><summary>Hint 1</summary>

```text
position of each assignment inside its expert's queue: one-hot the
flattened indices [T*k, E], cumsum over dim 0, read your own column.
```

</details>

<details><summary>Hint 2</summary>

```text
onehot = F.one_hot(indices.flatten(), E); pos = (onehot.cumsum(0) * onehot).sum(-1) - 1
kept = (pos < capacity).view(T, k)
```

</details>

<details><summary>Hint 3</summary>

```text
dropped assignments must contribute nothing: dispatch_loop with the
weights set to 0 where not kept gives y (an expert may then run on a
token it ignores; real kernels skip it, the result is the same).
```

</details>

In [ ]:
def dispatch_with_capacity(x: torch.Tensor, weights: torch.Tensor, indices: torch.Tensor,
                           experts: nn.ModuleList, capacity: int):
    """Like dispatch_loop, but every expert keeps at most `capacity` assignments.

    Assignments are served in token-major order: (token 0, slot 0), (token 0, slot 1),
    (token 1, slot 0), ...  An expert keeps the first `capacity` it receives and
    drops the rest. Returns (y [T, d], kept [T, k] bool).
    """
    raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(8, globals(), verbose=True)`.

In [ ]:
check.run_lab(8, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try the routers here (Run the cells above first).
logits = torch.randn(6, 4)              # 6 tokens, 4 experts
w, idx, probs = softmax_topk_router(logits, k=2)
print(idx)
print(load_balancing_loss(probs, idx))
expert_capacity(6, 4, 2, 1.25)